# Grouped time-series leakage lab

Original, fully synthetic data. Compare a frozen group/time split against two deliberately invalid evaluations. This is an educational example, not a device-failure model.

Data: CC0. Code: MIT. Author account: irynavs. No real device, medical or competition data.

[Generator and tests](https://github.com/siriushex/kaggle-validation-lab). The public holdout is for demonstration only; it is not a secret competition test set.

In [ ]:
from pathlib import Path
import importlib.util, json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn
candidates = list(Path('/kaggle/input').rglob('benchmark.py')) if Path('/kaggle/input').exists() else []
generator = candidates[0] if candidates else Path('benchmark.py')
spec = importlib.util.spec_from_file_location('benchmark', generator)
benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(benchmark)
csv_path = generator.parent / 'observations.csv'
data = pd.read_csv(csv_path) if csv_path.exists() else benchmark.generate()
print({'rows': len(data), 'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__})
data.groupby('split').size()

## 1. Validate the deployment contract

New devices at later times are the intended deployment setting. Fit imputation, scaling and encoding on training rows only. Identifiers and post-outcome values are excluded from deployable features.

In [ ]:
parts = {s: data[data['split'].eq(s)] for s in ['train', 'validation', 'holdout']}
assert data.row_id.is_unique
assert not set(parts['train'].device_id) & set(parts['validation'].device_id)
assert not set(parts['train'].device_id) & set(parts['holdout'].device_id)
assert pd.to_datetime(parts['train'].timestamp).max() < pd.to_datetime(parts['validation'].timestamp).min()
assert pd.to_datetime(parts['validation'].timestamp).max() < pd.to_datetime(parts['holdout'].timestamp).min()
assert 'post_outcome_score' not in benchmark.FEATURES
print('Contract checks passed')

## 2. Run predeclared comparisons

The honest pipeline uses the same fixed configuration for validation and one holdout evaluation. The intentionally leaky experiments demonstrate why good-looking numbers may be invalid. Do not tune configuration by rerunning the holdout.

In [ ]:
results = benchmark.evaluate(data)
table = pd.DataFrame({k: v for k, v in results.items() if isinstance(v, dict) and 'roc_auc' in v}).T
print(table.round(4).to_string())
Path('validation-results.json').write_text(json.dumps(results, indent=2))
repro = {'source_sha256': hashlib.sha256(generator.read_bytes()).hexdigest(), 'seed': benchmark.SEED, 'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__}
Path('environment.json').write_text(json.dumps(repro, indent=2))

## 3. Interpret before optimising

A near-perfect score from the post-outcome feature is a known bug, not a breakthrough. Random row splitting shares device identities and ignores the deployment horizon. Random and frozen evaluations also cover different populations, so their difference is not a causal estimate of leakage.

Suggested exercises: remove one valid feature; measure calibration; plot missingness; add a predeclared temporal embargo. Use new independent data for model selection. Synthetic results do not establish performance on real hardware or Kaggle competitions.